In [2]:
import sys
from pathlib import Path

# 노트북이 juypter-notebook/ 에 있으면 parent 한 번
sys.path.insert(0, str(Path.cwd().parent))
from docling.document_converter import DocumentConverter

# 변환기 생성 (PDF, 이미지 등 여러 포맷 지원)
converter = DocumentConverter()
print("DocumentConverter 준비 완료")

pdf_path = Path("../1_quotation_sample/26818_삼양철강 견적서 2022.05.11.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {pdf_path}")

result = converter.convert(pdf_path)

# display(Markdown(markdown_text))

# doc_dict = result.document.export_to_dict()

# doc_dict

DocumentConverter 준비 완료


In [3]:
# 마크다운을 렌더링해서 이쁘게 보기
from IPython.display import display, Markdown

markdown_text = result.document.export_to_markdown()
display(Markdown(markdown_text))

NO. 202205110001003

## 견적서

## (주)마켓오브메테리얼 貴中

울산광역시 북구 효문동 813-5

(대표) 052-288-8880

윤주선

F a x  :

프로젝트 :

수    신 :

참    조 :

의뢰번호 :

인도조건 :

지불조건 :

납기일자 :

유효기간 :

대표 이사 :

052-288-8880 (직통)

E-mail :

작성자 : 김진우

권 형우 팀장님

T e l  :

VAT포함 :  일금(￦9,276,998)

一金구백이십칠만육천구백구십팔 원정

합계금액 :

페이지 :1

Fax:

2022-05-11

견적 일자 :

<!-- image -->

<!-- image -->

|   순번 NO. | 품명 및 재질 규격 및 메이커   | 길 이 Length   | 수 량 Q'TY   | 단위 UNIT   | 중 량 WEIGHT   | 단위 UNIT   | 단 가 UNIT PRICE   |   금 액 AMOUNT |   비 고 REMARKS |
|------------|-------------------------------|----------------|--------------|-------------|----------------|-------------|--------------------|----------------|-----------------|
|        001 | PIPE SUS304(ERW) 6″ x SCH10S  | 6              | 9            | 본          | 739.8          | 본          | 512,694            |      4,614,246 |                 |
|        002 | PIPE SUS304(ERW) 8″ x SCH10S  | 6              | 1            | 본          | 127.2          | 본          | 740,500            |        740,500 |                 |
|        003 | PIPE SUS304(ERW) 10″ x SCH10S | 6              | 2            | 본          | 314.4          | 본          | 1,030,240          |      2,060,480 |                 |
|        004 | PIPE SUS304(ERW) 16″ x SCH10S | 2              | 1            | 본          | 94.6           | 본          | 375,400            |        375,400 |                 |
|        005 | ㄱ형강 65*65*6 SS275/A36      | 10             | 8            | 본          | 472.8          | KG          | 1,360              |        643,008 |                 |
|            |                               | 이             | 하           | 하          | 여             |             | 백                 |                |                 |
|            | 합 계 / 부 가 세              |                | 21           |             | 1,748.8        |             | 8,433,634          |                |         843,364 |

## 특기사항:

- *4번항 SCH10(6.35T) 생산불가, SCH10S(4.78T) 로 견적 합니다.
- *5번항 ㄱ형강 60*5는 수급불가로 65*6으로 대체견적합니다.

In [4]:
doc_dict = result.document.export_to_dict()
doc_dict.keys()

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])

In [5]:
len(doc_dict["groups"])
doc_dict["groups"][1].keys()

dict_keys(['self_ref', 'parent', 'children', 'content_layer', 'name', 'label'])

In [6]:
print(doc_dict["texts"][1].keys())
print()
print(doc_dict["texts"][1])
print(doc_dict["texts"][15])

dict_keys(['self_ref', 'parent', 'children', 'content_layer', 'label', 'prov', 'orig', 'text', 'level'])

{'self_ref': '#/texts/1', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'section_header', 'prov': [{'page_no': 1, 'bbox': {'l': 245.28, 't': 798.33264, 'r': 330.45768105599996, 'b': 770.25264, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 3]}], 'orig': '견적서', 'text': '견적서', 'level': 1}
{'self_ref': '#/texts/15', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 1, 'bbox': {'l': 329.76, 't': 687.12864, 'r': 386.63899200000003, 'b': 677.04864, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 7]}], 'orig': '대표 이사 :', 'text': '대표 이사 :'}


In [7]:
for group in doc_dict["groups"]:
    print(group["name"])
    print(group["self_ref"])

    for d in group["children"]:
        ref = d["$ref"]

        if ref.startswith("#/"):
            ref = ref[2:]
        elif ref.startswith("/"):
            ref = ref[1:]

    cur = doc_dict
    for p in ref.split("/"):
        cur = cur[int(p)] if p.isdigit() else cur[p]
    print(cur["text"])

    print()

group
#/groups/0
합계금액 :

group
#/groups/1
Fax:

group
#/groups/2
견적 일자 :

list
#/groups/3
*5번항 ㄱ형강 60*5는 수급불가로 65*6으로 대체견적합니다.



## 1. Document 타입 분류
* 키워드 매칭을 통해 scoring 하여 분류
* IBM opensource library `DoclingParser` 사용
  * 내부적으로 OCR, PDF loader 모두 지원
  * Vision AI 모델을 내포하고 있음

In [8]:
doc_dict = result.document.export_to_dict()
from document_parsing_engine.app.services import DocumentClassificationService

doc_service = DocumentClassificationService()
classification_result = doc_service.classify(doc_dict)
print("doc_type:", classification_result.doc_type)
print("score:", classification_result.score)
print(
    "reasons:",
    classification_result.reasons[:5] if classification_result.reasons else [],
)

doc_type: DocType.QUOTATION
score: 44
reasons: ['title exact match(section_header): "견적서"', 'text contains "견적": "견적서"', 'text contains "견적": "견적 일자 :"', 'text contains "견적": "*4번항 SCH10(6.35T) 생산불가, SCH10S(4.78T) 로 견적 합니다."', 'text contains "견적": "*5번항 ㄱ형강 60*5는 수급불가로 65*6으로 대체견적합니다."']


## 2. Document Layout Parsing
* doclingParser는 문서를 영역 별로 읽음
  * 문서의 유사한 정보는 __한 곳에 모여 있고 근처에 있다고 판단함__
  * 모여있는 텍스트를 하나의 block으로 인식
  *  block 기본적으로 단순 `text`, `group`, `table`, `picture` 로 나뉨
    * group은 `list`, `key-value` 로 나뉨

In [9]:
from document_parsing_engine.app.services import DocumentLayoutParsingService

svc = DocumentLayoutParsingService(y_tol=10, min_gap_threshold=18, gap_ratio=1.8)
# blocks_sorted = svc.get_blocks_sorted(doc_dict)

# # 그룹 ref별 known_keys (선택). 비우면 기본 파싱만 사용
# group_known_keys_map = {}  # 예: {"#/groups/0": {"견적 일자", "수신", "합계금액"}}

# # 구조화된 결과만 필요할 때
# blocks = svc.parse_blocks(
#     doc_dict, group_known_keys_map=group_known_keys_map, blocks_sorted=blocks_sorted
# )
# for blk in blocks:
#     print(blk.ref, blk.content_type, blk.content)  # 노트북처럼 출력만 할 때
# svc.print_blocks_content(
#     doc_dict, group_known_keys_map=group_known_keys_map, blocks_sorted=blocks_sorted
# )

In [10]:
blocks_sorted = svc.get_blocks_sorted(doc_dict)

# 그룹 ref별 known_keys (선택). 비우면 기본 파싱만 사용
group_known_keys_map = {}  # 예: {"#/groups/0": {"견적 일자", "수신", "합계금액"}}

# 구조화된 결과만 필요할 때
blocks = svc.parse_blocks(doc_dict)

# 노트북처럼 출력만 할 때
svc.print_blocks_content(
    doc_dict, group_known_keys_map=group_known_keys_map, blocks_sorted=blocks_sorted
)


[0] #/texts/0  label=text
NO. 202205110001003

[1] #/texts/1  label=section_header
견적서

[2] #/texts/2  label=section_header
(주)마켓오브메테리얼 貴中

[3] #/groups/0  label=key_value_area
    - 수 신: 권 형우 팀장님
    - 참 조: 
    - 프로젝트: 
    - 대표 이사: 윤주선
    - 의뢰번호: 울산광역시 북구 효문동 813-5
    - 인도조건: 
    - T e l: (대표) 052-288-8880
    - 지불조건: 
    - F a x: 052-288-8880 (직통)
    - 납기일자: 작성자 : 김진우
    - 유효기간: E-mail :
    - VAT포함: 일금(￦9,276,998)
    - 합계금액: 一金구백이십칠만육천구백구십팔 원정

[4] #/groups/1  label=key_value_area
    - Tel: 
    - Fax: 
    - 페이지: 1

[5] #/groups/2  label=key_value_area
    - 견적 일자: 2022-05-11

[6] #/pictures/0  label=picture
    [PICTURE]

[7] #/pictures/1  label=picture
    [PICTURE]

[8] #/tables/0  label=table
    row[0] = ['순번 NO.', '품명 및 재질 규격 및 메이커', '길 이 Length', "수 량 Q'TY", '단위 UNIT', '중 량 WEIGHT', '단위 UNIT', '단 가 UNIT PRICE', '금 액 AMOUNT', '비 고 REMARKS']
    row[1] = ['001', 'PIPE SUS304(ERW) 6″ x SCH10S', '6', '9', '본', '739.8', '본', '512,694', '4,614,246', '']
    row[2] = ['0

In [11]:
# svc.parse_blocks(doc_dict)

## 3. Segment Layout mapping
* 견적서는 `quotation_meta`, `items`, `remarks` 같은 세그먼트로 나뉨
* 위에서 parsing 된 블럭이 어떤 세그먼트인지 매핑해야함
* LLM이 필드 해석을 하기 위한 깨끗한 정보를 주기 위해서임
  
  * `ex prompt`
    * quotation_meta 란 철강 견적서에서 제공자, 고객, 일자 등이 포함된 세그먼트야.
    * quotation_meta 세그먼트에 추측되는 정보들로 아래와 같은 같은 정보들이 있어. 여기서 ~~ 데이터를 뽑아줘
    * RAG 활용시 - 철강 도메인지식 문서 제공

``` bash
quotation_meta
├── document_key
├── supplier_name
├── supplier_contact_person
├── supplier_phone
├── supplier_address
├── buyer_name
├── buyer_contact_person
├── buyer_phone
├── buyer_address
├── validity # (견적 유효기간)
├── payment_terms # (대금지급 조건)
└── delivery_terms # (운송 조건)

items
├── rows[]
│   ├── unit_price # (단가)
│   ├── quantity 
│   ├── line_amount # (단가 X 수량) 특정 항목에 대한 순수 금액 - 부가세, 배송비, 할인 제외)
│   ├── vat
│   ├── material_grade # (재질)
│   ├── thickness # (두께)
│   ├── size_text # (사이즈)
│   └── spec_text
└── totals
    ├── supply_amount
    └── total_amount
    
remarks
├── raw_text
└── extracted_conditions(optional)
```

In [12]:
from document_parsing_engine.app.services.layout_segment_mapping_service import (
    LayoutSegmentMappingService,
)

mapper = LayoutSegmentMappingService()
seg = mapper.recommend(
    doc_type=classification_result.doc_type,
    doc_dict={},
    blocks=blocks,
)
mapper.pretty_print(seg, blocks)

=== segment: quotation_meta ===
  block_refs: ['#/texts/1', '#/groups/0', '#/groups/1']
  reasons: ['segment keywords matched: 견적서', 'title keywords matched: 견적서', 'content_type=text']...
    - #/texts/1 | label=section_header | content_type=text
      견적서
    - #/groups/0 | label=key_value_area | content_type=group_kv
      수 신: 권 형우 팀장님
      참 조: 
      프로젝트: 
      대표 이사: 윤주선
      의뢰번호: 울산광역시 북구 효문동 813-5
      인도조건: 
      T e l: (대표) 052-288-8880
      지불조건: 
      F a x: 052-288-8880 (직통)
      납기일자: 작성자 : 김진우
      유효기간: E-mail :
      VAT포함: 일금(￦9,276,998)
      합계금액: 一金구백이십칠만육천구백구십팔 원정
    - #/groups/1 | label=key_value_area | content_type=group_kv
      Tel: 
      Fax: 
      페이지: 1

=== segment: items ===
  block_refs: ['#/tables/0']
  reasons: ["field keywords matched: unit price, price, q'ty, amount, 재질, length, 규격, 메이커", 'segment keywords matched: 품명, 재질, 규격', 'content_type=table']...
    - #/tables/0 | label=table | content_type=table
       ['순번 NO.', '품명 및 재질 규격 및 메

## 4. Segment Reconstruction
* 견적서는 `quotation_meta`, `items`, `remarks` 같은 세그먼트로 나뉨
* 아래와 같은 경우 segment 를 재구성 할 필요가 있음
  * 보통 quotation_meta는 `key-value` 형태 (T e l: (대표) 052-288-8880) 와 같은 형태이나 `table(표)` 로 된 경우
  * 반대로 items는 `table(표)` 형태이나 `key-value` 형태나 표 구조가 뭉개져 `text` 형태로 나눠져 있는 경우
* quotation_meta : table-> key-value
* items          : key-value, text -> table

In [12]:
### 개발 필요

## 5. Field Mapping
* Antrophic Sonnet 4.6 사용
* 문서에서 1차로 걸러진 필드 정보와 필드 수식 정보로 매핑을 추론해야하는데, 소넷이 추론 성능이 좋아서 사용해봄
* Gemini 2.5 flash API 같은 무료 모델로 해도 성능이 잘 나올거라 예상


### 5.1 사전에 정의된 필드정보
* 데이터베이스에서 관리하며 업데이트가능

In [16]:
from document_parsing_engine.domain.field_mapping.keywords import build_segment_field_keywords
build_segment_field_keywords(classification_result.doc_type.value)

{'quotation_meta': {'document_key': ['견적번호', 'no.', '문서번호', 'document no'],
  'supplier_name': ['공급업체', 'supplier', '판매처', '상호', '회사명'],
  'supplier_contact_person': ['담당자', '작성자', 'contact person'],
  'supplier_phone': ['tel', '전화', '연락처', 'phone', 'fax', 'email', 'e-mail'],
  'supplier_address': ['주소', 'address'],
  'buyer_name': ['수신', '귀중', '貴中', 'buyer', '고객사'],
  'buyer_contact_person': ['수신', '참조', '담당자'],
  'buyer_phone': ['tel', '전화', '연락처'],
  'buyer_address': ['주소', 'address'],
  'validity': ['유효기간', 'validity'],
  'payment_terms': ['지불조건', '결제조건', 'payment'],
  'delivery_terms': ['인도조건', '운송조건', 'delivery', '납기일자']},
 'items': {'rows': {'keywords': [],
   'unit_price': ['단가', 'unit price', 'price'],
   'quantity': ['수량', 'qty', "q'ty", 'quantity'],
   'line_amount': ['금액', 'amount'],
   'material_grade': ['재질', 'grade', 'material'],
   'thickness': ['두께', 'thickness'],
   'size_text': ['사이즈', 'size', '길이', '폭', 'length'],
   'spec_text': ['규격', 'spec', 'maker', '메이커', 'desc

### 5.2 세그먼트로 부터 추출된 데이터

In [19]:
from document_parsing_engine.domain.field_mapping.segment_data import build_segment_data
build_segment_data(seg, blocks)

{'quotation_meta': [{'ref': '#/texts/1',
   'label': 'section_header',
   'content_type': 'text',
   'content': '견적서'},
  {'ref': '#/groups/0',
   'label': 'key_value_area',
   'content_type': 'group_kv',
   'content': {'수 신': '권 형우 팀장님',
    '참 조': '',
    '프로젝트': '',
    '대표 이사': '윤주선',
    '의뢰번호': '울산광역시 북구 효문동 813-5',
    '인도조건': '',
    'T e l': '(대표) 052-288-8880',
    '지불조건': '',
    'F a x': '052-288-8880 (직통)',
    '납기일자': '작성자 : 김진우',
    '유효기간': 'E-mail :',
    'VAT포함': '일금(￦9,276,998)',
    '합계금액': '一金구백이십칠만육천구백구십팔 원정'}},
  {'ref': '#/groups/1',
   'label': 'key_value_area',
   'content_type': 'group_kv',
   'content': {'Tel': '', 'Fax': '', '페이지': '1'}}],
 'items': [{'ref': '#/tables/0',
   'label': 'table',
   'content_type': 'table',
   'content': [['순번 NO.',
     '품명 및 재질 규격 및 메이커',
     '길 이 Length',
     "수 량 Q'TY",
     '단위 UNIT',
     '중 량 WEIGHT',
     '단위 UNIT',
     '단 가 UNIT PRICE',
     '금 액 AMOUNT',
     '비 고 REMARKS'],
    ['001',
     'PIPE SUS304(ERW) 6″ x 

### 5.3 프롬프트
* `/Users/waonderboy/mom-document-parsing/document_parsing_engine/prompt/extraction_prompts.yaml` 참조

In [14]:
from document_parsing_engine.app.services.field_mapping_service import FieldMappingService

svc = FieldMappingService()
svc.extract(classification_result.doc_type.value, seg, blocks)

{'quotation_meta': {'document_key': 'NO. 202205110001003',
  'supplier_name': '',
  'supplier_contact_person': '김진우',
  'supplier_phone': '(대표) 052-288-8880',
  'supplier_address': '울산광역시 북구 효문동 813-5',
  'buyer_name': '(주)마켓오브메테리얼',
  'buyer_contact_person': '권 형우 팀장님',
  'buyer_phone': '',
  'buyer_address': '',
  'validity': '',
  'payment_terms': '',
  'delivery_terms': ''},
 'items': {'rows': [{'unit_price': 512694,
    'quantity': 9,
    'line_amount': 4614246,
    'vat': None,
    'material_grade': 'SUS304',
    'thickness': 'SCH10S',
    'size_text': '6″',
    'spec_text': 'PIPE SUS304(ERW) 6″ x SCH10S'},
   {'unit_price': 740500,
    'quantity': 1,
    'line_amount': 740500,
    'vat': None,
    'material_grade': 'SUS304',
    'thickness': 'SCH10S',
    'size_text': '8″',
    'spec_text': 'PIPE SUS304(ERW) 8″ x SCH10S'},
   {'unit_price': 1030240,
    'quantity': 2,
    'line_amount': 2060480,
    'vat': None,
    'material_grade': 'SUS304',
    'thickness': 'SCH10S',
    'siz

## 최종 정리
1. 오픈 소스 라이브러리(DoclingParser)로 문서 로딩
2. 레이아웃 파싱 및 좌표에 따라 정보를 그룹화 및 정렬
3. 그룹화 된 데이터를 견적서 세그먼트에 매핑 -> 견적서는 A,B,C 로 구성되니 데이터를 A,B,C에 맞게 거시적인 필터링을 해야함
4. 필드 매핑 -> 세그먼트 필드 메타 데이터와 세그먼트에 매핑된 그룹화 된 데이터를 LLM을 통해 매핑

## 다음 과제
* Soft Filter (세그먼트)
* key-value 그룹 고도화
* Validation
* Review queue


In [3]:
import sys
from pathlib import Path

# 노트북이 juypter-notebook/ 에 있으면 parent 한 번
sys.path.insert(0, str(Path.cwd().parent))
from document_parsing_engine.engine.document_parsing_engine import DocumentParsingEngine

engine = DocumentParsingEngine()
engine.process("../1_quotation_sample/26818_삼양철강 견적서 2022.05.11.pdf")



{'quotation_meta': {'document_key': 'NO. 202205110001003',
  'supplier_name': '',
  'supplier_contact_person': '김진우',
  'supplier_phone': '(대표) 052-288-8880',
  'supplier_address': '울산광역시 북구 효문동 813-5',
  'buyer_name': '(주)마켓오브메테리얼',
  'buyer_contact_person': '권 형우 팀장님',
  'buyer_phone': '',
  'buyer_address': '',
  'validity': '',
  'payment_terms': '',
  'delivery_terms': ''},
 'items': {'rows': [{'unit_price': 512694,
    'quantity': 9,
    'line_amount': 4614246,
    'vat': None,
    'material_grade': 'SUS304',
    'thickness': 'SCH10S',
    'size_text': '6″',
    'spec_text': 'PIPE SUS304(ERW) 6″ x SCH10S'},
   {'unit_price': 740500,
    'quantity': 1,
    'line_amount': 740500,
    'vat': None,
    'material_grade': 'SUS304',
    'thickness': 'SCH10S',
    'size_text': '8″',
    'spec_text': 'PIPE SUS304(ERW) 8″ x SCH10S'},
   {'unit_price': 1030240,
    'quantity': 2,
    'line_amount': 2060480,
    'vat': None,
    'material_grade': 'SUS304',
    'thickness': 'SCH10S',
    'siz

In [4]:
engine.process("../1_quotation_sample/13149_Quotation to Nicky-seamless pipe-Centerway _version 1_.pdf")

{'quotation_meta': {'document_key': 'CTWQT25005ZYH',
  'supplier_name': 'CENTERWAY STEEL CO., LTD',
  'supplier_contact_person': 'Zachary Zheng',
  'supplier_phone': '0086-731-86452692',
  'supplier_address': 'No.246 Shidai Yangguang Blvd, Yuhua District, Changsha, Hunan, China.',
  'buyer_name': 'Steelboso',
  'buyer_contact_person': 'Nlcky',
  'buyer_phone': '',
  'buyer_address': '',
  'validity': '5 days',
  'payment_terms': '30% T/T in advance, and the 70% balance before shipment (other terms negotiable)',
  'delivery_terms': '20 days after get the prepayment'},
 'items': {'rows': [{'unit_price': 325.65,
    'quantity': 1,
    'line_amount': 325.7,
    'vat': None,
    'material_grade': 'API 5L GR-B',
    'thickness': '5.08',
    'size_text': '48.3',
    'spec_text': 'seamless pipe API 5L GR-B PLS2'},
   {'unit_price': 286.78,
    'quantity': 2,
    'line_amount': 573.6,
    'vat': None,
    'material_grade': 'API 5L GR-B',
    'thickness': '4.55',
    'size_text': '33.4',
    'sp

In [5]:
engine.process("../1_quotation_sample/27188_1.견적요청서.pdf")

{'quotation_meta': {'document_key': 'KTL_KDC_002',
  'supplier_name': '',
  'supplier_contact_person': '',
  'supplier_phone': '',
  'supplier_address': '',
  'buyer_name': '마켓오브메테리얼',
  'buyer_contact_person': '김경호 이사님',
  'buyer_phone': '',
  'buyer_address': '',
  'validity': '',
  'payment_terms': '',
  'delivery_terms': '납품일 별도 회신 요청'},
 'items': {'rows': [{'unit_price': None,
    'quantity': 16,
    'line_amount': None,
    'vat': None,
    'material_grade': 'A105',
    'thickness': 'Sch.40',
    'size_text': '4"',
    'spec_text': 'ASEM #150 A105 Sch.40 WN.RF 4" (MFG : 펠릭스테크)'}],
  'totals': {'supply_amount': None, 'total_amount': None}},
 'remarks': {'raw_text': '1. 세금계산서 발행메일 info@tkdk.co.kr 2. 화물 납품 대신화물/택배 부산장안점 3. 택배 및 직납시 부산 기장군 장안읍 명례산단1로 166 3. 성적서(EN10204-3.1) 또는 테스트 리포트 는 반드시 발행해야하며, Mail : tech@tkdk.co.kr 로 회신 요청 4. 납품요청일에 따른 지체상환금은 납품요청일로부터 7일 이후 적용되며 1일에 3/1000(0.3%)가 적용. 이에 반드시 납품일 기입요청',
  'extracted_conditions': ['납품요청일에 따른 지체상환금은 납품요청일로부터 7일 이후 적용되며 1일에 3/1000(0